# GEC-RAG Project
## Notebook 06 — GEC-RAG Pipeline

This notebook implements the **GEC-RAG system** — the core contribution of the paper.

**Pipeline:**
1. Correct the input sentence using GPT-4o
2. Identify error types using ERRANT
3. Retrieve relevant pedagogical explanation from the KB using BM25
4. Generate a corpus-grounded explanation using GPT-4o + retrieved context

**Key difference from Notebook 05:**
The explanation is no longer generic — it is grounded in real learner error patterns from the Write & Improve Corpus 2024 via RAG retrieval.

**What this notebook produces:**
- GEC-RAG corrections and grounded explanations
- F0.5 score via ERRANT
- Per error-type performance breakdown
- Explanation quality data for G-Eval in Notebook 07
- Saved predictions for Notebook 07

---
### Cell 1 — Imports & Setup

In [1]:
import os
import json
import time
import pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
import errant
import warnings
warnings.filterwarnings('ignore')

BASE_DIR           = 'C:/gecrag'
DATA_DIR           = os.path.join(BASE_DIR, 'data/write_and_improve')
FINAL_VERSIONS_DIR = os.path.join(DATA_DIR, 'user-prompt-final-versions')
RESULTS_DIR        = os.path.join(BASE_DIR, 'results')
KB_DIR             = os.path.join(BASE_DIR, 'knowledge_base')
os.makedirs(RESULTS_DIR, exist_ok=True)

load_dotenv(os.path.join(BASE_DIR, '.env'))
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Say OK'}],
    max_tokens=5
)
print(f'\u2705 OpenAI API   : {test.choices[0].message.content}')
print(f'\u2705 Imports OK')

✅ OpenAI API   : OK!
✅ Imports OK


---
### Cell 2 — Load KB & BM25 Index

In [2]:
kb_path = os.path.join(KB_DIR, 'knowledge_base.json')
with open(kb_path, 'r', encoding='utf-8') as f:
    knowledge_base = json.load(f)
print(f'\u2705 KB loaded: {len(knowledge_base)} entries')

index_path = os.path.join(KB_DIR, 'bm25_index.pkl')
with open(index_path, 'rb') as f:
    index_data = pickle.load(f)

bm25_index        = index_data['bm25']
index_error_types = index_data['error_types']
print(f'\u2705 BM25 index loaded: {len(index_error_types)} entries')

annotator = errant.load('en')
print(f'\u2705 ERRANT loaded')

print(f'\nKB error types available:')
for et in index_error_types:
    freq = knowledge_base[et]['frequency']
    tier = knowledge_base[et]['tier']
    print(f'  [Tier {tier}] {et:<20} (freq: {freq:,})')

✅ KB loaded: 20 entries
✅ BM25 index loaded: 20 entries
✅ ERRANT loaded

KB error types available:
  [Tier 1] R:NOUN               (freq: 8,069)
  [Tier 1] R:ORTH               (freq: 4,089)
  [Tier 1] M:DET                (freq: 3,273)
  [Tier 1] R:SPELL              (freq: 3,647)
  [Tier 1] R:NOUN:NUM           (freq: 1,800)
  [Tier 1] R:PREP               (freq: 3,337)
  [Tier 1] R:VERB:TENSE         (freq: 1,938)
  [Tier 1] R:VERB               (freq: 2,453)
  [Tier 1] R:VERB:FORM          (freq: 2,022)
  [Tier 1] R:DET                (freq: 1,903)
  [Tier 2] R:MORPH              (freq: 1,270)
  [Tier 2] U:DET                (freq: 2,562)
  [Tier 2] M:PREP               (freq: 1,469)
  [Tier 2] R:VERB:SVA           (freq: 1,440)
  [Tier 2] U:PREP               (freq: 945)
  [Tier 2] M:VERB               (freq: 919)
  [Tier 2] R:PRON               (freq: 889)
  [Tier 2] M:PRON               (freq: 827)
  [Tier 2] R:WO                 (freq: 657)
  [Tier 2] R:ADJ                (freq

---
### Cell 3 — Load Dev Set

In [3]:
def load_text_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

orig_path = os.path.join(
    FINAL_VERSIONS_DIR,
    'en-writeandimprove2024-final-versions-dev-sentences.orig'
)
corr_path = os.path.join(
    FINAL_VERSIONS_DIR,
    'en-writeandimprove2024-final-versions-dev-sentences.corr'
)

orig_sentences = load_text_file(orig_path)
corr_sentences = load_text_file(corr_path)
assert len(orig_sentences) == len(corr_sentences)

print(f'\u2705 Dev split loaded: {len(orig_sentences)} sentence pairs')
print(f'\nSample:')
for i in range(3):
    print(f'  [{i+1}] ORIG: {orig_sentences[i]}')
    print(f'       CORR: {corr_sentences[i]}')

✅ Dev split loaded: 4352 sentence pairs

Sample:
  [1] ORIG: There are lots of important things in our life.
       CORR: There are lots of important things in our life.
  [2] ORIG: However, in most cases, they depend on the priorities we have and acceptation we are awaiting from it.
       CORR: However, in most cases, they depend on the priorities we have and the acceptation we are awaiting from it.
  [3] ORIG: If it fulfils our dreams and aims or not?
       CORR: If it fulfils our dreams and aims or not.


---
### Cell 4 — Define RAG Retrieval Functions

In [4]:
def retrieve_kb_entry(error_type, bm25, error_types, knowledge_base, top_k=1):
    query  = error_type.lower().replace(':', ' ')
    tokens = query.split()
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    for idx in top_idx:
        if scores[idx] > 0:
            et = error_types[idx]
            return {
                'error_type' : et,
                'score'      : round(float(scores[idx]), 3),
                'explanation': knowledge_base[et]['explanation'],
                'examples'   : knowledge_base[et]['examples'][:2],
                'frequency'  : knowledge_base[et]['frequency'],
            }
    return None


def detect_errors(original, corrected, annotator):
    try:
        orig_p = annotator.parse(original)
        corr_p = annotator.parse(corrected)
        edits  = annotator.annotate(orig_p, corr_p)
        return [
            {'type': e.type, 'original': e.o_str, 'correction': e.c_str}
            for e in edits if e.type != 'noop'
        ]
    except Exception:
        return []


# Test retrieval
test_types = ['M:DET', 'R:VERB:TENSE', 'R:NOUN:NUM', 'R:PREP']
print('Retrieval test:')
for et in test_types:
    result = retrieve_kb_entry(et, bm25_index, index_error_types, knowledge_base)
    if result:
        print(f'  Query : {et}')
        print(f'  Match : {result["error_type"]} (score: {result["score"]})')
        print(f'  Preview: {result["explanation"][:100]}...\n')
    else:
        print(f'  Query : {et} -> No match\n')

Retrieval test:
  Query : M:DET
  Match : M:DET (score: 3.738)
  Preview: 1. ERROR TYPE: Missing Articles

2. WHAT IT IS: This error involves leaving out necessary articles l...

  Query : R:VERB:TENSE
  Match : R:VERB:TENSE (score: 5.251)
  Preview: 1. ERROR TYPE: Verb Tense Confusion

2. WHAT IT IS: This error involves using the wrong verb tense i...

  Query : R:NOUN:NUM
  Match : R:NOUN:NUM (score: 5.28)
  Preview: 1. ERROR TYPE: Singular and Plural Noun Confusion

2. WHAT IT IS: This error occurs when learners mi...

  Query : R:PREP
  Match : R:PREP (score: 2.699)
  Preview: 1. ERROR TYPE: Preposition Confusion

2. WHAT IT IS: This error involves using the wrong preposition...



---
### Cell 5 — Define GEC-RAG Pipeline

In [5]:
GEC_SYSTEM_PROMPT = (
    'You are a grammatical error correction assistant.\n'
    'Your task is to correct grammatical errors in English sentences written by L2 learners.\n\n'
    'For each sentence you receive, respond with a JSON object containing:\n'
    '- "corrected": the corrected sentence (if no errors, return the original unchanged)\n\n'
    'Respond ONLY with the JSON object. No preamble or extra text.'
)

RAG_EXPLANATION_TEMPLATE = (
    'You are an expert EFL teacher providing feedback to an L2 English learner.\n\n'
    'A learner wrote the following sentence:\n'
    '  Original  : {original}\n'
    '  Corrected : {corrected}\n\n'
    'The following grammatical errors were detected:\n'
    '{error_list}\n\n'
    'Here is relevant pedagogical knowledge about these error types, grounded in real learner data:\n'
    '{kb_context}\n\n'
    'Using the pedagogical knowledge above as context, write a clear, helpful explanation for the learner.\n'
    'The explanation should:\n'
    '1. Identify what was corrected\n'
    '2. Explain why it was wrong using the corpus-grounded knowledge provided\n'
    '3. Give a clear rule the learner can remember\n'
    '4. Be encouraging and accessible to a B1-B2 level learner\n\n'
    'Write only the explanation -- no JSON, no preamble.'
)


def run_gecrag_pipeline(sentence, client, annotator, bm25, error_types, knowledge_base):
    result = {
        'original'        : sentence,
        'corrected'       : sentence,
        'errors_detected' : [],
        'kb_retrieved'    : [],
        'explanation'     : None,
        'rag_used'        : False
    }

    # Step 1 -- GEC correction
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': GEC_SYSTEM_PROMPT},
                {'role': 'user',   'content': f'Correct this sentence:\n\n{sentence}'}
            ],
            max_tokens=200,
            temperature=0
        )
        raw    = response.choices[0].message.content.strip()
        clean  = raw.replace('```json','').replace('```','').strip()
        parsed = json.loads(clean)
        result['corrected'] = parsed.get('corrected', sentence)
    except Exception:
        return result

    if result['corrected'] == sentence:
        return result

    # Step 2 -- Error detection
    errors = detect_errors(sentence, result['corrected'], annotator)
    result['errors_detected'] = errors
    if not errors:
        return result

    # Step 3 -- RAG retrieval
    kb_entries = []
    seen_types = set()
    for error in errors:
        et = error['type']
        if et in seen_types or et == 'noop':
            continue
        seen_types.add(et)
        kb_entry = retrieve_kb_entry(et, bm25, error_types, knowledge_base)
        if kb_entry:
            kb_entries.append(kb_entry)
    result['kb_retrieved'] = kb_entries

    # Step 4 -- Generate grounded explanation
    if kb_entries:
        result['rag_used'] = True
        error_list = '\n'.join([
            f'  - [{e["type"]}] "{e["original"]}" -> "{e["correction"]}"'
            for e in errors
        ])
        kb_context = ''
        for entry in kb_entries[:2]:
            kb_context += f'\n[{entry["error_type"]}]\n{entry["explanation"]}\n'
        prompt = RAG_EXPLANATION_TEMPLATE.format(
            original=sentence, corrected=result['corrected'],
            error_list=error_list, kb_context=kb_context
        )
        try:
            expl = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=400, temperature=0.3
            )
            result['explanation'] = expl.choices[0].message.content.strip()
        except Exception:
            pass

    return result


# Sanity check
print('Testing GEC-RAG pipeline...')
test_result = run_gecrag_pipeline(
    'I go to school yesterday and learned many things.',
    client, annotator, bm25_index, index_error_types, knowledge_base
)
sep = '='*60
print(f'\n{sep}')
print(f'Original  : {test_result["original"]}')
print(f'Corrected : {test_result["corrected"]}')
print(f'Errors    : {[e["type"] for e in test_result["errors_detected"]]}')
print(f'KB used   : {[e["error_type"] for e in test_result["kb_retrieved"]]}')
print(f'RAG used  : {test_result["rag_used"]}')
print(f'\nExplanation:')
print(test_result['explanation'])
print(sep)

Testing GEC-RAG pipeline...

Original  : I go to school yesterday and learned many things.
Corrected : I went to school yesterday and learned many things.
Errors    : ['R:VERB:TENSE']
KB used   : ['R:VERB:TENSE']
RAG used  : True

Explanation:
Your original sentence was: "I go to school yesterday and learned many things." The correction made was changing "go" to "went." 

The reason for this correction is that you used the present tense verb "go" when you were talking about something that happened in the past—yesterday. In English, we need to match the verb tense with the time of the action. Since you are referring to an action that has already happened, you should use the past tense "went."

A helpful rule to remember is: **Use the past tense for actions that happened in the past.** So, when you talk about something that occurred yesterday, always use the past tense form of the verb.

Keep practicing, and you'll get even better at using the correct tenses! You're doing great!


---
### Cell 6 — Run GEC-RAG on Dev Set
> ⚠️ Set MAX_SENTENCES=50 for quick test first.
> ⏳ Full run (~4352 sentences) costs approximately $8-12 in API credits.

In [10]:
def run_gecrag_batch(sentences, client, annotator, bm25, error_types, knowledge_base, max_sentences=None):
    n         = max_sentences or len(sentences)
    sentences = sentences[:n]
    results   = []
    errors    = 0
    for sent in tqdm(sentences, desc='GEC-RAG pipeline'):
        try:
            result = run_gecrag_pipeline(
                sent, client, annotator, bm25, error_types, knowledge_base
            )
            results.append(result)
            time.sleep(0.1)
        except Exception:
            results.append({
                'original': sent, 'corrected': sent,
                'errors_detected': [], 'kb_retrieved': [],
                'explanation': None, 'rag_used': False
            })
            errors += 1
    rag_used_count = sum(1 for r in results if r['rag_used'])
    print(f'\n\u2705 GEC-RAG complete')
    print(f'   Processed     : {len(results)}')
    print(f'   RAG triggered : {rag_used_count} ({rag_used_count/len(results)*100:.1f}%)')
    print(f'   Errors        : {errors}')
    return results


# ⚠️ Change to None for full run
MAX_SENTENCES = None

print(f'Running GEC-RAG on {MAX_SENTENCES or len(orig_sentences)} sentences...')
gecrag_results = run_gecrag_batch(
    orig_sentences, client, annotator,
    bm25_index, index_error_types, knowledge_base,
    max_sentences=MAX_SENTENCES
)

print('\nSample results (with RAG):')
rag_examples = [r for r in gecrag_results if r['rag_used']][:2]
for r in rag_examples:
    sep = '-'*55
    print(f'\n{sep}')
    print(f'Original  : {r["original"]}')
    print(f'Corrected : {r["corrected"]}')
    print(f'Errors    : {[e["type"] for e in r["errors_detected"]]}')
    print(f'KB entries: {[e["error_type"] for e in r["kb_retrieved"]]}')
    print(f'\nExplanation:')
    print(r['explanation'])

Running GEC-RAG on 4352 sentences...


GEC-RAG pipeline:   0%|          | 0/4352 [00:00<?, ?it/s]


✅ GEC-RAG complete
   Processed     : 4352
   RAG triggered : 3740 (85.9%)
   Errors        : 0

Sample results (with RAG):

-------------------------------------------------------
Original  : There are lots of important things in our life.
Corrected : There are lots of important things in our lives.
Errors    : ['R:NOUN']
KB entries: ['R:NOUN:NUM']

Explanation:
Your original sentence was: "There are lots of important things in our life." It was corrected to: "There are lots of important things in our lives."

The correction was made because "life" should be in the plural form "lives" in this context. When you say "our," you are referring to multiple people (you and others), so you need to use the plural form to match that idea. 

In English, nouns can change depending on whether they refer to one thing or more than one. In this case, since you are talking about the lives of multiple people, the correct form is "lives." This is a common error for learners because it can be tricky to 

---
### Cell 7 — Compute F0.5 Score

In [11]:
def compute_f05(orig_list, pred_list, ref_list, annotator):
    tp = fp = fn = 0
    for orig, pred, ref in tqdm(
        zip(orig_list, pred_list, ref_list),
        total=len(orig_list), desc='Computing F0.5'
    ):
        try:
            orig_p = annotator.parse(orig)
            pred_p = annotator.parse(pred)
            ref_p  = annotator.parse(ref)
            pred_edits = set(
                (e.o_start, e.o_end, e.c_str)
                for e in annotator.annotate(orig_p, pred_p) if e.type != 'noop'
            )
            ref_edits = set(
                (e.o_start, e.o_end, e.c_str)
                for e in annotator.annotate(orig_p, ref_p) if e.type != 'noop'
            )
            tp += len(pred_edits & ref_edits)
            fp += len(pred_edits - ref_edits)
            fn += len(ref_edits  - pred_edits)
        except Exception:
            continue
    beta      = 0.5
    precision = tp/(tp+fp) if (tp+fp) > 0 else 0
    recall    = tp/(tp+fn) if (tp+fn) > 0 else 0
    f05       = ((1+beta**2)*precision*recall /
                 (beta**2*precision+recall)) if (precision+recall) > 0 else 0
    return {
        'precision': round(precision*100, 2),
        'recall'   : round(recall*100,    2),
        'f05'      : round(f05*100,       2),
        'tp': tp, 'fp': fp, 'fn': fn
    }


n_gecrag     = len(gecrag_results)
gecrag_preds = [r['corrected'] for r in gecrag_results]
ref_gecrag   = corr_sentences[:n_gecrag]

print('Computing F0.5 for GEC-RAG...')
gecrag_scores = compute_f05(
    orig_sentences[:n_gecrag], gecrag_preds, ref_gecrag, annotator
)

sep = '='*45
print(f'\n{sep}')
print('GEC-RAG Results')
print(sep)
print(f'  Precision : {gecrag_scores["precision"]}%')
print(f'  Recall    : {gecrag_scores["recall"]}%')
print(f'  F0.5      : {gecrag_scores["f05"]}%')
print(f'  TP/FP/FN  : {gecrag_scores["tp"]}/{gecrag_scores["fp"]}/{gecrag_scores["fn"]}')
print(sep)

Computing F0.5 for GEC-RAG...


Computing F0.5:   0%|          | 0/4352 [00:00<?, ?it/s]


GEC-RAG Results
  Precision : 37.34%
  Recall    : 54.01%
  F0.5      : 39.8%
  TP/FP/FN  : 4422/7420/3765


---
### Cell 8 — Per Error-Type Performance

In [12]:
def compute_per_type_scores(orig_list, pred_list, ref_list, annotator):
    type_stats = {}
    for orig, pred, ref in tqdm(
        zip(orig_list, pred_list, ref_list),
        total=len(orig_list), desc='Per-type analysis'
    ):
        try:
            orig_p = annotator.parse(orig)
            pred_p = annotator.parse(pred)
            ref_p  = annotator.parse(ref)
            pred_edits = {
                (e.o_start, e.o_end, e.c_str): e.type
                for e in annotator.annotate(orig_p, pred_p) if e.type != 'noop'
            }
            ref_edits = {
                (e.o_start, e.o_end, e.c_str): e.type
                for e in annotator.annotate(orig_p, ref_p) if e.type != 'noop'
            }
            for key, etype in ref_edits.items():
                if etype not in type_stats:
                    type_stats[etype] = {'tp':0,'fp':0,'fn':0}
                if key in pred_edits:
                    type_stats[etype]['tp'] += 1
                else:
                    type_stats[etype]['fn'] += 1
            for key, etype in pred_edits.items():
                if key not in ref_edits:
                    if etype not in type_stats:
                        type_stats[etype] = {'tp':0,'fp':0,'fn':0}
                    type_stats[etype]['fp'] += 1
        except Exception:
            continue
    beta = 0.5
    results = []
    for etype, s in type_stats.items():
        p   = s['tp']/(s['tp']+s['fp']) if (s['tp']+s['fp']) > 0 else 0
        r   = s['tp']/(s['tp']+s['fn']) if (s['tp']+s['fn']) > 0 else 0
        f05 = ((1+beta**2)*p*r/(beta**2*p+r)) if (p+r) > 0 else 0
        results.append({
            'error_type': etype,
            'tp': s['tp'], 'fp': s['fp'], 'fn': s['fn'],
            'precision' : round(p*100,   2),
            'recall'    : round(r*100,   2),
            'f05'       : round(f05*100, 2),
        })
    return pd.DataFrame(results).sort_values('f05', ascending=False)


print('Computing per error-type scores...')
gecrag_per_type = compute_per_type_scores(
    orig_sentences[:n_gecrag], gecrag_preds, ref_gecrag, annotator
)
specific = gecrag_per_type[~gecrag_per_type['error_type'].str.contains('OTHER')]
print('\nGEC-RAG Per Error-Type F0.5 (top 15, OTHER excluded):')
print(specific.head(15).to_string(index=False))

Computing per error-type scores...


Per-type analysis:   0%|          | 0/4352 [00:00<?, ?it/s]


GEC-RAG Per Error-Type F0.5 (top 15, OTHER excluded):
 error_type  tp  fp  fn  precision  recall   f05
R:NOUN:INFL   2   0   1     100.00   66.67 90.91
    R:SPELL 357 113  79      75.96   81.88 77.07
 R:VERB:SVA 156  81  25      65.82   86.19 69.09
M:VERB:FORM  54  29  12      65.06   81.82 67.84
R:VERB:FORM 212 153 108      58.08   66.25 59.55
     R:ORTH 430 317 286      57.56   60.06 58.05
     R:PART  38  31  24      55.07   61.29 56.21
      M:DET 217 182 150      54.39   59.13 55.27
 R:NOUN:NUM 141 153  80      47.96   63.80 50.47
R:NOUN:POSS   1   1   1      50.00   50.00 50.00
    R:MORPH  88 100  53      46.81   62.41 49.27
     M:PRON  69  78  54      46.94   56.10 48.52
     R:PREP 266 315 225      45.78   54.18 47.25
      U:DET 178 217 133      45.06   57.23 47.07
 R:ADJ:FORM   4   5   3      44.44   57.14 46.51


---
### Cell 9 — RAG Coverage Analysis

In [13]:
total           = len(gecrag_results)
corrected       = sum(1 for r in gecrag_results if r['corrected'] != r['original'])
rag_triggered   = sum(1 for r in gecrag_results if r['rag_used'])
no_correction   = total - corrected
corrected_no_kb = corrected - rag_triggered

kb_counter = {}
for r in gecrag_results:
    for entry in r['kb_retrieved']:
        et = entry['error_type']
        kb_counter[et] = kb_counter.get(et, 0) + 1

sep = '='*55
print(sep)
print('RAG COVERAGE ANALYSIS')
print(sep)
print(f'\n\U0001f4ca Pipeline Statistics:')
print(f'   Total sentences      : {total}')
print(f'   Corrections made     : {corrected} ({corrected/total*100:.1f}%)')
print(f'   No correction needed : {no_correction} ({no_correction/total*100:.1f}%)')
print(f'   RAG triggered        : {rag_triggered} ({rag_triggered/total*100:.1f}%)')
print(f'   Corrected w/o KB hit : {corrected_no_kb}')
print(f'\n\U0001f4ca Most Retrieved KB Entries:')
print(f'   {"Error Type":<20} {"Times Retrieved":>16}')
print(f'   {"-"*38}')
for et, count in sorted(kb_counter.items(), key=lambda x: -x[1]):
    print(f'   {et:<20} {count:>16}')

RAG COVERAGE ANALYSIS

📊 Pipeline Statistics:
   Total sentences      : 4352
   Corrections made     : 3740 (85.9%)
   No correction needed : 612 (14.1%)
   RAG triggered        : 3740 (85.9%)
   Corrected w/o KB hit : 0

📊 Most Retrieved KB Entries:
   Error Type            Times Retrieved
   --------------------------------------
   R:VERB:SVA                       2262
   R:NOUN:NUM                       1811
   U:PREP                            676
   M:PRON                            642
   R:ORTH                            622
   R:PREP                            512
   M:DET                             448
   R:VERB:TENSE                      429
   U:DET                             424
   R:SPELL                           415
   R:VERB:FORM                       393
   R:DET                             266
   M:PREP                            229
   R:ADJ                             198
   R:MORPH                           182
   R:PRON                            121
   R:WO   

---
### Cell 10 — Save Results

In [14]:
serializable_results = []
for r in gecrag_results:
    serializable_results.append({
        'original'        : r['original'],
        'corrected'       : r['corrected'],
        'errors_detected' : r['errors_detected'],
        'kb_retrieved'    : [
            {'error_type': e['error_type'], 'score': e['score'], 'frequency': e['frequency']}
            for e in r['kb_retrieved']
        ],
        'explanation'     : r['explanation'],
        'rag_used'        : r['rag_used']
    })

gecrag_path = os.path.join(RESULTS_DIR, 'gecrag_predictions.json')
with open(gecrag_path, 'w', encoding='utf-8') as f:
    json.dump({
        'system'      : 'GEC-RAG (GPT-4o-mini + KB)',
        'model'       : 'gpt-4o-mini + BM25 + KB',
        'n_sentences' : len(gecrag_results),
        'scores'      : gecrag_scores,
        'predictions' : serializable_results
    }, f, ensure_ascii=False, indent=2)
print(f'\u2705 GEC-RAG predictions saved : {gecrag_path}')

per_type_path = os.path.join(RESULTS_DIR, 'gecrag_per_type_scores.csv')
gecrag_per_type.to_csv(per_type_path, index=False)
print(f'\u2705 Per-type scores saved     : {per_type_path}')

scores_path = os.path.join(RESULTS_DIR, 'baseline_scores.json')
try:
    with open(scores_path, 'r') as f:
        all_scores = json.load(f)
except FileNotFoundError:
    all_scores = {}
all_scores['GEC-RAG (GPT-4o-mini + KB)'] = gecrag_scores
with open(scores_path, 'w') as f:
    json.dump(all_scores, f, indent=2)
print(f'\u2705 Baseline scores updated   : {scores_path}')

✅ GEC-RAG predictions saved : C:/gecrag\results\gecrag_predictions.json
✅ Per-type scores saved     : C:/gecrag\results\gecrag_per_type_scores.csv
✅ Baseline scores updated   : C:/gecrag\results\baseline_scores.json


---
### Cell 11 — Full System Comparison

In [15]:
scores_path = os.path.join(RESULTS_DIR, 'baseline_scores.json')
with open(scores_path, 'r') as f:
    all_scores = json.load(f)

sep = '='*65
print(sep)
print('FULL SYSTEM COMPARISON -- CORRECTION QUALITY (F0.5)')
print(sep)
print(f'\n  {"System":<35} {"Precision":>10} {"Recall":>8} {"F0.5":>8}')
print(f'  {"-"*63}')
for system, scores in all_scores.items():
    marker = ' <' if 'RAG' in system else ''
    print(
        f'  {system:<35} '
        f'{scores["precision"]:>9}% '
        f'{scores["recall"]:>7}% '
        f'{scores["f05"]:>7}%{marker}'
    )
print(f'\n  < = Our proposed system')
print(f'\n\u26a0\ufe0f  Reminder: If you ran with MAX_SENTENCES=50,')
print(f'   set MAX_SENTENCES=None in Cell 6 and re-run cells 6-10.')
print(f'\n\U0001f4c1 Output Files:')
print(f'   - results/gecrag_predictions.json')
print(f'   - results/gecrag_per_type_scores.csv')
print(f'   - results/baseline_scores.json')
print(f'\n\u27a1\ufe0f  Next: Run Notebook 07 -- Evaluation & G-Eval')

FULL SYSTEM COMPARISON -- CORRECTION QUALITY (F0.5)

  System                               Precision   Recall     F0.5
  ---------------------------------------------------------------
  T5-GEC                                  48.51%   25.19%   40.93%
  GPT-4o-mini (no RAG)                     37.4%   53.73%   39.82% <
  Llama-3.1-8B (no RAG)                   31.17%   43.79%   33.08% <
  GEC-RAG (GPT-4o-mini + KB)              37.34%   54.01%    39.8% <

  < = Our proposed system

⚠️  Reminder: If you ran with MAX_SENTENCES=50,
   set MAX_SENTENCES=None in Cell 6 and re-run cells 6-10.

📁 Output Files:
   - results/gecrag_predictions.json
   - results/gecrag_per_type_scores.csv
   - results/baseline_scores.json

➡️  Next: Run Notebook 07 -- Evaluation & G-Eval
